# Ekstraliga 2025 — Exploratory Data Analysis
This notebook explores the PA Logs dataset before modelling.

In [2]:
import sys
sys.path.insert(0, 'src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

from preprocessing import build_pa_dataframe
from features import (
    compute_batter_tendencies, compute_pitcher_tendencies,
    build_feature_matrix, train_val_test_split, OUTCOME_CATEGORIES
)
from models.markov import OutcomeMarkov, PitchTransitionMarkov

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

ModuleNotFoundError: No module named 'matplotlib'

## 1. Load data

In [ ]:
XLSX = 'data/ekstraliga/2025_Ekstraliga_Stats.xlsx'
pa_df = build_pa_dataframe(XLSX)
print(pa_df.shape)
pa_df.head()

## 2. Outcome distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw result counts
pa_df['result'].value_counts().head(15).plot.bar(ax=axes[0], color='steelblue')
axes[0].set_title('Top 15 PA Result Tokens')
axes[0].set_xlabel('Result Token')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

# Coarse category counts
cat_counts = pa_df['result_category'].value_counts()
cat_counts.plot.pie(ax=axes[1], autopct='%1.1f%%',
                    colors=sns.color_palette('tab10', len(cat_counts)))
axes[1].set_title('Outcome Category Distribution')
axes[1].set_ylabel('')

plt.tight_layout()
plt.savefig('notebooks/outcome_distribution.png', dpi=150)
plt.show()

## 3. Sequence length distribution

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
pa_df['seq_len'].value_counts().sort_index().plot.bar(ax=ax, color='coral')
ax.set_title('Plate Appearance Length (# pitches)')
ax.set_xlabel('Pitches in PA')
ax.set_ylabel('Count')
plt.tight_layout()
plt.savefig('notebooks/seq_length_dist.png', dpi=150)
plt.show()

print(pa_df['seq_len'].describe())

## 4. Pitch token frequencies

In [ ]:
all_tokens = [tok for seq in pa_df['sequence'] for tok in seq]
token_counts = Counter(all_tokens)

fig, ax = plt.subplots(figsize=(12, 4))
labels, vals = zip(*sorted(token_counts.items(), key=lambda x: -x[1]))
ax.bar(labels, vals, color='teal')
ax.set_title('Pitch Token Frequency (all pitches in all PAs)')
ax.set_ylabel('Count')
plt.tight_layout()
plt.savefig('notebooks/token_freq.png', dpi=150)
plt.show()

## 5. Markov transition matrix

In [ ]:
markov = PitchTransitionMarkov(smoothing=0.1)
markov.fit(pa_df)

trans = markov.transition_matrix_df()
# Show subset of most common tokens
show_tokens = ['B', 'F', 'Sw', 'Sc', 'Ks', 'Kc', 'BB', 'GO', 'FO', '1B', '2B', 'HR']
trans_sub = trans.loc[[t for t in show_tokens if t in trans.index],
                       [t for t in show_tokens if t in trans.columns]]

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(trans_sub, annot=True, fmt='.2f', cmap='YlOrRd',
            linewidths=0.5, ax=ax)
ax.set_title('Marginalised Pitch Transition Probabilities')
ax.set_xlabel('Next pitch')
ax.set_ylabel('Current pitch')
plt.tight_layout()
plt.savefig('notebooks/markov_transition.png', dpi=150)
plt.show()

## 6. Top batters and pitchers by tendency

In [ ]:
bt = compute_batter_tendencies(pa_df)
pt = compute_pitcher_tendencies(pa_df)

# Minimum 20 PA to appear
bt_q = bt[bt['batter_n_pa'] >= 20]
pt_q = pt[pt['pitcher_n_pa'] >= 20]

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

bt_q.nlargest(10, 'hit_rate')['hit_rate'].plot.barh(ax=axes[0,0], color='green')
axes[0,0].set_title('Top Batters: Hit Rate (min 20 PA)')

bt_q.nlargest(10, 'k_rate')['k_rate'].plot.barh(ax=axes[0,1], color='red')
axes[0,1].set_title('Highest K% Batters')

pt_q.nlargest(10, 'pitcher_k_rate')['pitcher_k_rate'].plot.barh(ax=axes[1,0], color='navy')
axes[1,0].set_title('Top Pitchers: K Rate')

pt_q.nsmallest(10, 'pitcher_hit_allow')['pitcher_hit_allow'].plot.barh(ax=axes[1,1], color='purple')
axes[1,1].set_title('Best Pitchers: Lowest Hit-Allow Rate')

plt.tight_layout()
plt.savefig('notebooks/player_tendencies.png', dpi=150)
plt.show()

## 7. Quick Markov evaluation

In [ ]:
feat_df = build_feature_matrix(pa_df, bt, pt)
train_df, val_df, test_df = train_val_test_split(feat_df)

outcome_markov = OutcomeMarkov(smoothing=0.5)
outcome_markov.fit(train_df)

val_m  = outcome_markov.evaluate(val_df)
test_m = outcome_markov.evaluate(test_df)

print(f'Markov Val  Accuracy: {val_m["accuracy"]:.4f}')
print(f'Markov Test Accuracy: {test_m["accuracy"]:.4f}')

# Majority-class baseline
most_common = train_df['result_category'].mode()[0]
baseline_acc = (test_df['result_category'] == most_common).mean()
print(f'Majority-class baseline: {baseline_acc:.4f} ({most_common})')